In [4]:
import os
import numpy as np
import cv2
from sklearn.model_selection import train_test_split

# --- SETTINGS ---
DATA_FOLDER = "data/UTKFace"   # folder where your images are
IMG_SIZE    = 64                # resize all faces to 64x64 pixels
SAVE_FILE   = "data/dataset.npz"  # we save processed data here

def load_dataset(folder):
    images  = []
    ages    = []
    genders = []

    all_files = os.listdir(folder)
    print(f"Total files found: {len(all_files)}")

    for filename in all_files:
        # skip files that are not images
        if not filename.endswith(".jpg"):
            continue

        # --- parse age and gender from filename ---
        parts = filename.split("_")
        if len(parts) < 2:
            continue  # skip badly named files

        try:
            age    = int(parts[0])
            gender = int(parts[1])   # 0=Male, 1=Female
        except ValueError:
            continue  # skip if parsing fails

        # skip unrealistic ages
        if age < 1 or age > 116:
            continue

        # --- load and resize the image ---
        img_path = os.path.join(folder, filename)
        img = cv2.imread(img_path)

        if img is None:
            continue  # skip unreadable images

        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # convert BGR -> RGB

        images.append(img)
        ages.append(age)
        genders.append(gender)

    print(f"Successfully loaded: {len(images)} images")
    return np.array(images), np.array(ages), np.array(genders)


def save_dataset(images, ages, genders):
    # normalize pixel values from 0-255 to 0-1
    images = images.astype("float32") / 255.0

    # split into training (80%) and testing (20%)
    X_train, X_test, age_train, age_test, gen_train, gen_test = train_test_split(
        images, ages, genders,
        test_size=0.2,
        random_state=42
    )

    # save everything to a single file
    os.makedirs("data", exist_ok=True)
    np.savez_compressed(
        SAVE_FILE,
        X_train=X_train, X_test=X_test,
        age_train=age_train, age_test=age_test,
        gen_train=gen_train, gen_test=gen_test
    )
    print(f"\nDataset saved to: {SAVE_FILE}")
    print(f"Training samples : {len(X_train)}")
    print(f"Testing  samples : {len(X_test)}")


if __name__ == "__main__":
    images, ages, genders = load_dataset(DATA_FOLDER)
    save_dataset(images, ages, genders)
    print("\nStep 1 complete! Run step2_train_model.py next.")

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'data/UTKFace'

In [2]:
import os
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from model import build_model

DATA_DIR = "../data/UTKFace"
IMG_SIZE = 64

def load_data(data_dir):
    images, genders, ages = [], [], []
    for fname in os.listdir(data_dir):
        try:
            parts = fname.split("_")
            age, gender = int(parts[0]), int(parts[1])
            img = cv2.imread(os.path.join(data_dir, fname))
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            images.append(img)
            genders.append(gender)
            ages.append(age)
        except:
            continue
    return np.array(images) / 255.0, np.array(genders), np.array(ages)

X, y_gender, y_age = load_data(DATA_DIR)

# Normalize age to [0,1] for stable training
y_age_norm = y_age / 116.0

X_train, X_val, yg_train, yg_val, ya_train, ya_val = train_test_split(
    X, y_gender, y_age_norm, test_size=0.2, random_state=42
)

model = build_model()
model.compile(
    optimizer=Adam(1e-4),
    loss={'gender': 'binary_crossentropy', 'age': 'mae'},
    loss_weights={'gender': 1.0, 'age': 0.5},
    metrics={'gender': 'accuracy', 'age': 'mae'}
)

callbacks = [
    ModelCheckpoint("saved/best_model.keras", save_best_only=True, monitor='val_loss'),
    EarlyStopping(patience=10, restore_best_weights=True),
    ReduceLROnPlateau(patience=5, factor=0.5)
]

model.fit(
    X_train, {'gender': yg_train, 'age': ya_train},
    validation_data=(X_val, {'gender': yg_val, 'age': ya_val}),
    epochs=50,
    batch_size=64,
    callbacks=callbacks
)

ModuleNotFoundError: No module named 'model'